# `mutate` — Reference
`mutate` creates or overwrites columns using clean keyword argument syntax (`col="expr"` or `col=callable`). Each entry is evaluated in order via pandas `eval()` — a plain formula per column, or a lambda when needed.
### Calling Styles
1. **Keyword arguments** (most Pythonic): `.pt.mutate(bmi="body_mass_g / bill_length_mm ** 2", mass_kg="body_mass_g / 1000")`
2. **Dictionary unpacking** (for new column names with spaces): `.pt.mutate(**{"body mass kg": "body_mass_g / 1000"})`
3. **External file specification**: `.pt.mutate("@features.txt")`
---


In [1]:
import sys, os
_src = os.path.abspath(os.path.join(os.getcwd(), '..', '..', 'src'))
if not os.path.exists(_src): _src = os.path.abspath(os.path.join(os.getcwd(), '..', 'src'))
if _src not in sys.path: sys.path.insert(0, _src)

import numpy as np
import pytae as pt

penguins = pt.sample_data['penguins']

## A single derived column

In [2]:
# Body mass index style ratio — clean assignment formula via kwargs
(
    penguins
    .pt.mutate(bmi='body_mass_g / bill_length_mm ** 2')
    .pt.select('species', 'body_mass_g', 'bill_length_mm', 'bmi')
    .sample(10)
)


,species,body_mass_g,bill_length_mm,bmi
328,Gentoo,4575.0,43.3,2.440143
282,Gentoo,4400.0,45.7,2.106785
298,Gentoo,4750.0,45.2,2.324967
58,Adelie,2850.0,36.5,2.139238
101,Adelie,4725.0,41.0,2.810827
89,Adelie,3600.0,38.9,2.379049
163,Chinstrap,3775.0,51.7,1.412329
19,Adelie,4200.0,46.0,1.984877
226,Gentoo,4800.0,45.4,2.328786
130,Adelie,3325.0,38.5,2.243211


In [3]:
# Column names inside the expression must stay unquoted — quoting one turns it into
# a string literal, which will break arithmetic
try:
    penguins.pt.mutate(bmi="'body_mass_g' / 'bill_length_mm' ** 2")
except TypeError as exc:
    print(f"TypeError (as expected): {exc}")


TypeError (as expected): unsupported operand type(s) for ** or pow(): 'str' and 'int'


## Multiple entries in one call, and chaining a later entry off an earlier one
Entries are applied left to right, so a later expression can reference a column derived earlier in the *same* `mutate()` call.

In [4]:
# Two independent derived columns in one call via kwargs
(
    penguins
    .pt.mutate(heavy='body_mass_g > 4000', mass_kg='body_mass_g / 1000')
    .pt.select('species', 'body_mass_g', 'mass_kg', 'heavy')
    .sample(10)
)


,species,body_mass_g,mass_kg,heavy
119,Adelie,3325.0,3.325,False
125,Adelie,4000.0,4.000,False
63,Adelie,4050.0,4.050,True
120,Adelie,3150.0,3.150,False
94,Adelie,3300.0,3.300,False
224,Gentoo,5400.0,5.400,True
331,Gentoo,5950.0,5.950,True
47,Adelie,2975.0,2.975,False
102,Adelie,3075.0,3.075,False
180,Chinstrap,3700.0,3.700,False


In [5]:
# mass_lb references mass_kg, derived by the keyword just before it
(
    penguins
    .pt.mutate(mass_kg='body_mass_g / 1000', mass_lb='mass_kg * 2.20462')
    .pt.select('species', 'body_mass_g', 'mass_kg', 'mass_lb')
    .sample(10)
)


,species,body_mass_g,mass_kg,mass_lb
293,Gentoo,5200.0,5.200,11.464024
66,Adelie,3350.0,3.350,7.385477
261,Gentoo,5700.0,5.700,12.566334
4,Adelie,3450.0,3.450,7.605939
302,Gentoo,4725.0,4.725,10.416829
283,Gentoo,5650.0,5.650,12.456103
128,Adelie,3050.0,3.050,6.724091
224,Gentoo,5400.0,5.400,11.904948
153,Chinstrap,3900.0,3.900,8.598018
11,Adelie,3700.0,3.700,8.157094


## String comparisons and local variables
String literals *inside* the expression (e.g. `'Adelie'`) need real quotes — column names stay bare. A variable from the calling scope can be referenced with an `@` prefix, same as pandas' own `eval()`/`query()`.


In [6]:
# String comparisons inside the expression formula
(
    penguins
    .pt.mutate(is_adelie="species == 'Adelie'")
    .pt.select('species', 'is_adelie')
    .sample(10)
)


,species,is_adelie
15,Adelie,True
100,Adelie,True
37,Adelie,True
267,Gentoo,False
179,Chinstrap,False
131,Adelie,True
82,Adelie,True
252,Gentoo,False
134,Adelie,True
146,Adelie,True


In [7]:
# @-prefixed names resolve against the scope that called mutate(), not the module
# — works identically whether calling pt.mutate() or chaining .pt.mutate()
threshold = 4000

(
    penguins
    .pt.mutate(heavy='body_mass_g >= @threshold')
    .pt.select('species', 'body_mass_g', 'heavy')
    .sample(10)
)


,species,body_mass_g,heavy
318,Gentoo,4625.0,True
88,Adelie,3950.0,False
327,Gentoo,5500.0,True
223,Gentoo,5700.0,True
262,Gentoo,4300.0,True
154,Chinstrap,3650.0,False
58,Adelie,2850.0,False
119,Adelie,3325.0,False
56,Adelie,3550.0,False
7,Adelie,4675.0,True


## Overwriting an existing column

In [8]:
# mutate() can overwrite a column in place, e.g. converting units
(
    penguins
    .pt.mutate(body_mass_g='body_mass_g / 1000')
    .pt.select('species', 'body_mass_g')
    .sample(10)
)


,species,body_mass_g
68,Adelie,3.050
237,Gentoo,6.300
150,Adelie,3.700
306,Gentoo,4.600
156,Chinstrap,3.725
153,Chinstrap,3.900
45,Adelie,4.600
257,Gentoo,5.250
283,Gentoo,5.650
0,Adelie,3.750


## Column names with spaces — backtick quoting
`pandas.eval()` uses **backticks**, not the single/double quotes used elsewhere in pytae, to reference a column name containing a space.

In [9]:
import pandas as pd
spaced = pd.DataFrame({'body mass g': [3750, 4200], 'bill length mm': [39.1, 46.5]})

# Backticks protect column names that contain spaces
spaced.pt.mutate(bmi="`body mass g` / `bill length mm` ** 2")


,body mass g,bill length mm,bmi
0,3750,39.1,2.452888
1,4200,46.5,1.942421


## Real-world pipeline: mutate → filter → select
`mutate()` chains like any other pytae method — filter on a column you just derived with `qry()`.

In [10]:
(
    penguins
    .pt.mutate(bmi='body_mass_g / bill_length_mm ** 2')
    .pt.qry(bmi='> 2')
    .pt.select('species', 'island', 'body_mass_g', 'bill_length_mm', 'bmi')
    .sample(10)
)


,species,island,body_mass_g,bill_length_mm,bmi
110,Adelie,Biscoe,3825.0,38.1,2.635005
270,Gentoo,Biscoe,4850.0,46.6,2.233417
238,Gentoo,Biscoe,4800.0,46.2,2.248833
266,Gentoo,Biscoe,4200.0,45.5,2.028740
304,Gentoo,Biscoe,4750.0,44.9,2.356139
314,Gentoo,Biscoe,4850.0,44.5,2.449186
343,Gentoo,Biscoe,5400.0,49.9,2.168666
64,Adelie,Biscoe,2850.0,36.4,2.151008
250,Gentoo,Biscoe,5250.0,47.3,2.346589
299,Gentoo,Biscoe,5950.0,45.2,2.912327


## Conditional column creation — `if_else()`, `case_when()`, `map()`
Plain `eval()` has no ternary/`where()` support, so `mutate()` provides three dplyr-style helpers as ordinary function calls, evaluated via `np.where()`/`np.select()`/`Series.map()`: `if_else(condition, true_value, false_value)`, `case_when((cond1, val1), (cond2, val2), ..., default)`, and `map(column, {key: value, ...}, default)`. A trailing bare argument to `case_when` is the catch-all default (like SQL ELSE). Because they are ordinary calls, they compose and chain with each other and with any pandas method. String outcomes need quotes; conditions are vectorized — prefer `and`/`or`/`not` (the bitwise `&`/`|`/`~` also work).

In [11]:
# if_else(condition, true_value, false_value) — like dplyr's if_else()
# true_value and false_value can be scalars or column names; conditions are vectorized
(
    penguins
    .pt.mutate(weight_class="if_else(body_mass_g > 4000, 'heavy', 'light')")
    .pt.select('species', 'body_mass_g', 'weight_class')
    .sample(10)
)


,species,body_mass_g,weight_class
6,Adelie,3625.0,light
329,Gentoo,5500.0,heavy
286,Gentoo,4650.0,heavy
74,Adelie,3700.0,light
264,Gentoo,5550.0,heavy
86,Adelie,3800.0,light
87,Adelie,3500.0,light
34,Adelie,3325.0,light
136,Adelie,3175.0,light
79,Adelie,4000.0,light


In [12]:
# case_when supports tuples or flat pairs with default=
(
    penguins
    .pt.mutate(
        size_class="case_when(body_mass_g >= 4500, 'large', body_mass_g >= 3500, 'medium', default='small')"
    )
    .pt.select('species', 'body_mass_g', 'size_class')
    .sample(10)
)


,species,body_mass_g,size_class
72,Adelie,3550.0,medium
52,Adelie,3450.0,small
49,Adelie,4150.0,medium
149,Adelie,3750.0,medium
141,Adelie,3475.0,small
184,Chinstrap,3350.0,small
80,Adelie,3200.0,small
282,Gentoo,4400.0,medium
297,Gentoo,6000.0,large
300,Gentoo,4625.0,large


In [13]:
# map(column, {key: value, ...}, default) — recode a column through a dictionary lookup
# unmapped keys become default if given, else NaN
(
    penguins
    .pt.mutate(
        island_code="map(island, {'Torgersen': 'TOR', 'Biscoe': 'BIS'}, 'OTH')"
    )
    .pt.select('species', 'island', 'island_code')
    .sample(10)
)


,species,island,island_code
308,Gentoo,Biscoe,BIS
247,Gentoo,Biscoe,BIS
84,Adelie,Dream,OTH
96,Adelie,Dream,OTH
109,Adelie,Biscoe,BIS
193,Chinstrap,Dream,OTH
54,Adelie,Biscoe,BIS
271,Gentoo,Biscoe,BIS
187,Chinstrap,Dream,OTH
86,Adelie,Dream,OTH


## First non-null resolution — `coalesce()`

`coalesce(col1, col2, ..., default)` evaluates candidates left-to-right and returns the first non-null value per row, like SQL `COALESCE()` or `dplyr::coalesce()`:

In [14]:
contacts = pd.DataFrame({
    'mobile': [None, '555-1234', None],
    'home': ['555-5678', None, None],
    'work': [None, None, '555-9012'],
})

contacts.pt.mutate(preferred_contact="coalesce(mobile, home, work, 'N/A')")


,mobile,home,work,preferred_contact
0,NaN,555-5678,NaN,555-5678
1,555-1234,NaN,NaN,555-1234
2,NaN,NaN,555-9012,555-9012


## Combining expressions and callables in kwargs

`mutate()` cleanly mixes expression strings and python callables (e.g. lambdas) in keyword arguments:


In [15]:
(
    penguins
    .pt.mutate(
        bmi='body_mass_g / bill_length_mm ** 2',
        mass_kg='body_mass_g / 1000',
        is_heavy=lambda df: df['body_mass_g'] > 4000,
    )
    .pt.select('species', 'bmi', 'mass_kg', 'is_heavy')
    .head(5)
)


,species,bmi,mass_kg,is_heavy
0,Adelie,2.452888,3.75,False
1,Adelie,2.435507,3.80,False
2,Adelie,2.001121,3.25,False
3,Adelie,NaN,NaN,False
4,Adelie,2.561456,3.45,False


## Loading specs from an external file — `@specs.txt`
For complex feature engineering or shared pipelines across Python and the CLI, specs can be loaded from an external file. The file supports multiline expressions, `#` comments, empty lines, and column chaining (where later formulas reference columns defined earlier in the same file).
### File content of `penguin_features.txt`:
```text
# Engineering features for penguins
mass_kg = body_mass_g / 1000
# Convert kg to lbs (references mass_kg computed above)
mass_lb = mass_kg * 2.20462
```


In [16]:
# Write a small demo spec file with comments and assignment '=' syntax
spec_content = """# Engineering features for penguins
mass_kg = body_mass_g / 1000

# Convert kg to lbs
mass_lb = mass_kg * 2.20462
"""
with open("penguin_features.txt", "w") as f:
    f.write(spec_content)

# Load directly using @filename
result = penguins.pt.mutate("@penguin_features.txt")
out = (
    result
    .pt.select("species", "mass_kg", "mass_lb")
    .sample(5)
)

# Clean up demo file
import os
os.remove("penguin_features.txt")

out


,species,mass_kg,mass_lb
127,Adelie,4.300,9.479866
302,Gentoo,4.725,10.416829
105,Adelie,3.550,7.826401
331,Gentoo,5.950,13.117489
91,Adelie,4.300,9.479866


## Columns with spaces — SQL-style brackets `[col]`

SQL-style square brackets `[col a]` are supported alongside backticks, avoiding shell backtick substitution hazards:

In [17]:
spaced.pt.mutate(ratio='[body mass g] / [bill length mm]')


,body mass g,bill length mm,ratio
0,3750,39.1,95.907928
1,4200,46.5,90.322581


## Creating new columns with spaces — dictionary unpacking (`**{...}`)

Python keyword argument syntax requires valid identifier names (`key=value`), so bare names with spaces like `my col = "..."` cause a Python `SyntaxError`.

To create a new column whose name contains spaces, pass the column definition in a dictionary and unpack it using `**`:


In [18]:
# Create a new column with spaces using dictionary unpacking
(
    spaced
    .pt.mutate(**{'body mass ratio': '[body mass g] / [bill length mm]'})
)


,body mass g,bill length mm,body mass ratio
0,3750,39.1,95.907928
1,4200,46.5,90.322581


In [19]:
# Define multiple new columns with spaces at once
(
    penguins
    .pt.mutate(**{
        'body mass kg': 'body_mass_g / 1000',
        'bill ratio': 'bill_length_mm / bill_depth_mm',
    })
    .pt.select('species', 'body mass kg', 'bill ratio')
    .head()
)


,species,body mass kg,bill ratio
0,Adelie,3.75,2.090909
1,Adelie,3.80,2.270115
2,Adelie,3.25,2.238889
3,Adelie,NaN,NaN
4,Adelie,3.45,1.901554


Note: When loading specs from an external file (`@specs.txt`) or via the CLI (`-mutate`), names with spaces can be written directly on the left-hand side of `=` without dictionary unpacking:

```text
# inside an external spec file or CLI flag:
my bmi = [body mass g] / [bill length mm] ** 2
```
